In [ ]:
# Install missing
!pip install pandas pillow torch torchvision matplotlib tqdm



In [ ]:
# ======================================
# Import librerie e setup
# ======================================
import os
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

# Impostazioni Jupyter
%matplotlib inline
plt.style.use('seaborn-v0_8')



# Controllo device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device usato:", device)
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
class CupsDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        batch = str(self.df.loc[idx, "Batch"])          # FORZA stringa
        image_name = str(self.df.loc[idx, "Image"])     # FORZA stringa
        label = torch.tensor(self.df.iloc[idx]["ObjectsCount"], dtype=torch.float32)


        img_path = os.path.join(
            self.root_dir,
            batch,
            "non_etichettate",
            f"{batch}_{image_name}"
        )

        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Immagine non trovata: {img_path}")

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
# ======================================
# Modello Custom CNN : Regressore (In realtà verrà riscritto, serve giusto a ricordare come era fatto il modello) 
# ======================================
import torch
from torch import nn

class CustomCNN(nn.Module):
    def __init__(self, input_size=(3, 500, 500)):
        super(CustomCNN, self).__init__()

        # ----------------------------
        # Feature extractor
        # ----------------------------
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # /2 -> 500 -> 250
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # /2 -> 250 -> 125
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # /2 -> 125 -> 62
        )

        
        with torch.no_grad():
            dummy = torch.zeros(1, *input_size)  # batch 1, canali x H x W
            dummy = self.features(dummy)
            self.flatten_dim = dummy.numel()  # totale elementi dopo conv

        # ----------------------------
        # Classifier
        # ----------------------------
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)  # output regressione
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ----------------------------
# Spostamento su GPU (se disponibile)
# ----------------------------
model = CustomCNN().to(device)

In [ ]:
FINE_TUNE_DIR = "./finetuning"

fine_tune_df = pd.read_csv(os.path.join(FINE_TUNE_DIR, "objects_count.csv"))
print("Immagini reali per fine-tuning:", len(fine_tune_df))
fine_tune_df.head()


In [ ]:
#obiettivo calcolare media e dev stand
from torch.utils.data import DataLoader
from torchvision import transforms

stats_transform = transforms.Compose([
    transforms.Resize((500, 500)),  # stessa dimensione usata per training
    transforms.ToTensor(),           
])

stats_dataset = CupsDataset(
    fine_tune_df,
    root_dir=FINE_TUNE_DIR,
    transform=stats_transform
)

stats_loader = DataLoader(
    stats_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

mean = torch.zeros(3)
std = torch.zeros(3)
total_images = 0

for images, _ in stats_loader:
    batch_size = images.size(0)
    images = images.view(batch_size, 3, -1)

    mean += images.mean(dim=2).sum(dim=0)
    std += images.std(dim=2).sum(dim=0)
    total_images += batch_size

mean /= total_images
std /= total_images

print("Mean:", mean)
print("Std:", std)


In [ ]:
#da usare in training
transform = transforms.Compose([
    transforms.Resize((500, 500)),
    transforms.RandomRotation(5),
    transforms.RandomHorizontalFlip(0.3),
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.3,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Lambda(
        lambda x: torch.clamp(x + 0.01 * torch.randn_like(x), 0.0, 1.0)
    ),
    transforms.Normalize(
        mean=mean.tolist(),
        std=std.tolist()
    )
])

#da usare in inferenza
inference_transform = transforms.Compose([
    transforms.Resize((500, 500)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean.tolist(),
        std=std.tolist()
    )
])



In [ ]:
# ======================================
# Loss e ottimizzatore vecchi
# ======================================
#criterion = criterion = nn.MAELoss()
#optimizer_old = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
checkpoint_path = "./checkpoints/checkpoint_epoch_15.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
#optimizer_old.load_state_dict(checkpoint["optimizer_state_dict"])
train_losses = checkpoint.get("train_losses", [])
val_losses   = checkpoint.get("val_losses", [])


print(f"Checkpoint caricato (epoch {checkpoint['epoch']})")


In [ ]:
for param in model.features.parameters():
    param.requires_grad = False   # blocco tutto il backbone

for param in model.features[-1].parameters():  # ultimo conv attivo
    param.requires_grad = True    # sblocco solo l'ultimo layer convoluzionale


In [ ]:
#cambio optimizer
fine_tune_optimizer = torch.optim.Adam([
    {'params': model.features[-1].parameters(), 'lr': 1e-5},  # ultimo layer conv
    {'params': model.classifier.parameters(), 'lr': 1e-4}      # head
], weight_decay=1e-5)
criterion = nn.L1Loss()

In [ ]:
# ======================================
# Dataset BASE (senza transform)
# ======================================
from torch.utils.data import DataLoader, random_split
import torch

base_dataset = CupsDataset(
    dataframe=fine_tune_df,
    root_dir=FINE_TUNE_DIR,
    transform=None
)

# ======================================
# Split train / val / test (riproducibile)
# ======================================
total_len = len(base_dataset)

train_len = int(0.7 * total_len)
val_len   = int(0.15 * total_len)
test_len  = total_len - train_len - val_len

generator = torch.Generator().manual_seed(42)

train_ds, val_ds, test_ds = random_split(
    base_dataset,
    [train_len, val_len, test_len],
    generator=generator
)

# ======================================
# Applica le transform DOPO lo split
# ======================================
train_ds.dataset.transform = transform
val_ds.dataset.transform   = inference_transform
test_ds.dataset.transform  = inference_transform

# ======================================
# DataLoader
# ======================================
train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print(
    f"Dataset sizes -> "
    f"Train: {len(train_ds)}, "
    f"Val: {len(val_ds)}, "
    f"Test: {len(test_ds)}"
)


In [ ]:
start_epoch = checkpoint["epoch"] + 1  
total_epochs = 29  
checkpoint_dir = "./checkpoints"

In [ ]:
for epoch in range(start_epoch, total_epochs):
    print(f"\nEpoch {epoch}/{total_epochs}")

    # ========= TRAIN =========
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1).float()

        fine_tune_optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        fine_tune_optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)
    train_losses.append(train_loss)

   

    # ========= VALIDATION =========
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1).float()
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

    val_loss /= len(val_loader.dataset)
    val_losses.append(val_loss)


    # ----------------------
    # Stampa riepilogo
    # ----------------------
    print(
        f"Epoch {epoch} complete | "
        f"Train Loss: {train_loss:.4f}  | "
        f"Val Loss: {val_loss:.4f} "
    )

    print(
        f"GPU memory used: "
        f"{torch.cuda.memory_allocated(device)/1024**2:.1f} MB"
    )

    # ----------------------
    # Checkpoint a ogni epoca 
    # ----------------------
    checkpoint_path = os.path.join(
        checkpoint_dir, f"fineTune_checkpoint_epoch_{epoch}.pt"
    )

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': fine_tune_optimizer.state_dict(),
        'train_losses': train_losses,
        'val_losses': val_losses
    }, checkpoint_path)

    print(f"Checkpoint salvato: {checkpoint_path}")


In [ ]:
# ======================================
# Plot andamento loss
# ======================================
plt.figure(figsize=(8,5))
plt.plot(range(0,total_epochs), train_losses, label='Train Loss')
plt.plot(range(0,total_epochs), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MAE Loss')
plt.title('Andamento Loss durante il training')
plt.legend()
plt.show()

In [ ]:
import torch
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1).float()

        outputs = model(images)

        all_preds.append(outputs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

# concatena tutto
y_pred = np.vstack(all_preds).ravel()
y_true = np.vstack(all_labels).ravel()

# calcola metriche manualmente
mae  = np.mean(np.abs(y_true - y_pred))
mse  = np.mean((y_true - y_pred)**2)
rmse = np.sqrt(mse)

# R² manuale
ss_res = np.sum((y_true - y_pred)**2)
ss_tot = np.sum((y_true - np.mean(y_true))**2)
r2    = 1 - ss_res / ss_tot

print("Test set evaluation")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


In [ ]:
import torch
import numpy as np
from torchvision import transforms

# Definiamo le trasformazioni TTA
tta_transforms = [
    transforms.Compose([transforms.Resize((500, 500))]),  # versione base
    transforms.Compose([transforms.Resize((500, 500)),
                        transforms.RandomHorizontalFlip(p=1.0)]),
    transforms.Compose([transforms.Resize((500, 500)),
                        transforms.ColorJitter(brightness=0.2, contrast=0.2)]),
    transforms.Compose([transforms.Resize((500, 500)),
                        transforms.RandomRotation(5)])
]

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        labels = labels.unsqueeze(1).float()  # [batch, 1]
        for i in range(images.size(0)):
            image = images[i].to(device)
            preds = []

            # TTA: predici su più versioni augmentate
            for aug in tta_transforms:
                img_aug = aug(image.cpu())  # trasformazioni su CPU
                img_aug = img_aug.unsqueeze(0).to(device)  # batch dimension
                pred = model(img_aug)
                preds.append(pred.item())

            final_pred = np.mean(preds)
            all_preds.append(final_pred)
            all_labels.append(labels[i].item())

# Calcola metriche
y_pred = np.array(all_preds)
y_true = np.array(all_labels)

mae  = np.mean(np.abs(y_true - y_pred))
mse  = np.mean((y_true - y_pred)**2)
rmse = np.sqrt(mse)
r2    = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)

print("Test set evaluation (TTA)")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
